## Generates one incremental update file per dataset, simulating a second data release from the municipality.

In [1]:
import os
import sys
import random
import pandas as pd

def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()

    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)

    raise FileNotFoundError(
        f"Could not find project root (looking for {marker})"
    )


PROJECT_ROOT = find_project_root()

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)

RAW = os.path.join(PROJECT_ROOT, "data", "raw")
UPDATES = os.path.join(PROJECT_ROOT, "data", "updates")

random.seed(42)

from src.common.spark_session import get_spark_session
from datetime import timedelta

spark = get_spark_session()

print("Project root:", PROJECT_ROOT)
print("Working directory:", os.getcwd())
print("Raw path:", RAW)
print("Updates path:", UPDATES)

26/09/24 21:46:48 WARN Utils: Your hostname, Thilinis-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.8 instead (on interface en0)
26/09/24 21:46:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 21:46:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Project root: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform
Working directory: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform
Raw path: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/raw
Updates path: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/updates


### Check trapdata latest record

In [2]:
import os
import pyarrow.parquet as pq

PROJECT_ROOT = os.getcwd()
RAW = os.path.join(PROJECT_ROOT, "data", "raw")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Looking in:", os.path.join(RAW, "tripdata"))

src_files = sorted(
    os.path.join(RAW, "tripdata", f) for f in os.listdir(os.path.join(RAW, "tripdata"))
    if f.endswith(".parquet")
)
print("\ntripdata files found:")
for f in src_files:
    print(" ", f)

original = pq.read_table(src_files[-1]).to_pandas()

march_only = original[
    (original["tpep_pickup_datetime"].dt.year == 2024) &
    (original["tpep_pickup_datetime"].dt.month == 3)
]

print("rows in March 2024:", len(march_only))
print("earliest March pickup:", march_only["tpep_pickup_datetime"].min())
print("latest March pickup:  ", march_only["tpep_pickup_datetime"].max())

PROJECT_ROOT: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform
Looking in: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/raw/tripdata

tripdata files found:
  /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/raw/tripdata/yellow_tripdata_2024-01.parquet
  /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/raw/tripdata/yellow_tripdata_2024-02.parquet
  /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/raw/tripdata/yellow_tripdata_2024-03.parquet
rows in March 2024: 3582605
earliest March pickup: 2024-03-01 00:00:00
latest March pickup:   2024-03-31 23:59:59


### Tripdata update: 5-10% new rows + 1-2% duplicates

In [3]:
def generate_tripdata_update():
    src_files = sorted(
        os.path.join(RAW, "tripdata", f) for f in os.listdir(os.path.join(RAW, "tripdata"))
        if f.endswith(".parquet")
    )
    original = pq.read_table(src_files[-1]).to_pandas()  # use the latest month as the base

    last_ts = original["tpep_pickup_datetime"].max()

    # 5-10% new rows: resample real rows, shift timestamps forward
    pct_new = random.uniform(0.05, 0.10)
    n_new = int(len(original) * pct_new)
    new_rows = original.sample(n=n_new, replace=True, random_state=1).copy()

    time_offsets = pd.to_timedelta(
        [random.randint(1, 60 * 24 * 14) for _ in range(n_new)], unit="min"  # spread over next 2 weeks
    )
    new_rows["tpep_pickup_datetime"] = last_ts + time_offsets
    duration = (original["tpep_dropoff_datetime"] - original["tpep_pickup_datetime"]).sample(
        n=n_new, replace=True, random_state=2
    ).reset_index(drop=True)
    new_rows["tpep_dropoff_datetime"] = new_rows["tpep_pickup_datetime"].reset_index(drop=True) + duration

    # jitter fare and distance slightly so rows resemble but aren't identical to source rows
    new_rows["trip_distance"] = (new_rows["trip_distance"] * random.uniform(0.9, 1.1)).round(2)
    new_rows["fare_amount"] = (new_rows["fare_amount"] * random.uniform(0.9, 1.1)).round(2)

    # 1-2% exact duplicates copied verbatim from the ORIGINAL dataset
    pct_dup = random.uniform(0.01, 0.02)
    n_dup = int(len(original) * pct_dup)
    dup_rows = original.sample(n=n_dup, random_state=3).copy()

    update = pd.concat([new_rows, dup_rows], ignore_index=True)

    out_path = os.path.join(RAW, "tripdata", "tripdata_update_2024_04.parquet")
    update.to_parquet(out_path, index=False)
    print(f"tripdata update: {n_new} new rows, {n_dup} duplicate rows -> {out_path}")
    return out_path, n_new, n_dup

### Weather update: new hourly rows + new `humidity` column

In [4]:
def generate_weather_update():
    src_path = sorted(
        os.path.join(RAW, "weather", f) for f in os.listdir(os.path.join(RAW, "weather")) if f.endswith(".csv")
    )[-1]
    original = pd.read_csv(src_path)

    last = original.sort_values(["year", "month", "day", "hour"]).iloc[-1]
    start = pd.Timestamp(year=int(last.year), month=int(last.month), day=int(last.day), hour=int(last.hour))

    valid_coco_rows = original.dropna(subset=["coco"])
    if valid_coco_rows.empty:
        valid_coco_rows = original

    n_new = 24 * 7
    rows = []
    for i in range(1, n_new + 1):
        ts = start + timedelta(hours=i)
        base_row = valid_coco_rows.sample(n=1).iloc[0]
        rows.append({
            "year": ts.year, "month": ts.month, "day": ts.day, "hour": ts.hour,
            "temp": round(base_row["temp"] + random.uniform(-2, 2), 1),
            "rhum": round(base_row["rhum"] + random.uniform(-5, 5), 1),
            "prcp": round(max(0, base_row["prcp"] + random.uniform(-0.5, 0.5)), 2),
            "wdir": base_row["wdir"],
            "wspd": round(max(0, base_row["wspd"] + random.uniform(-2, 2)), 1),
            "pres": round(base_row["pres"] + random.uniform(-1, 1), 1),
            "coco": int(base_row["coco"]),   # <-- force plain int, avoid "5.0" string in the CSV
            "humidity": round(random.uniform(20, 100), 1),
        })

    update = pd.DataFrame(rows)
    out_path = os.path.join(RAW, "weather", "weather_update.csv")
    update.to_csv(out_path, index=False)
    print(f"weather update: {len(update)} new rows, +humidity column -> {out_path}")
    return out_path, len(update)

### Air quality update: new rows + new `aqi` column

In [5]:
def generate_air_quality_update():
    src_path = sorted(
        os.path.join(RAW, "air_quality", f) for f in os.listdir(os.path.join(RAW, "air_quality"))
        if f.endswith(".csv")
    )[-1]
    original = pd.read_csv(src_path)

    n_new = min(500, len(original) // 20)
    new_rows = original.sample(n=n_new, replace=True, random_state=4).copy()

    # shift Date and Time Local & GMT forward by 7 days as a simple "next period" simulation
    for date_col, time_col in [("Date Local", "Time Local"), ("Date GMT", "Time GMT")]:
        combined = pd.to_datetime(new_rows[date_col] + " " + new_rows[time_col]) + timedelta(days=7)
        new_rows[date_col] = combined.dt.strftime("%Y-%m-%d")
        new_rows[time_col] = combined.dt.strftime("%H:%M")

    new_rows["Sample Measurement"] = (new_rows["Sample Measurement"] * random.uniform(0.8, 1.2)).round(2)
    new_rows["aqi"] = [random.randint(0, 200) for _ in range(n_new)]

    out_path = os.path.join(RAW, "air_quality", "air_quality_update.csv")
    new_rows.to_csv(out_path, index=False)
    print(f"air_quality update: {len(new_rows)} new rows, +aqi column -> {out_path}")
    return out_path, len(new_rows)

### Run all three generators

In [6]:
tripdata_path, tripdata_n_new, tripdata_n_dup = generate_tripdata_update()
weather_path, weather_n_new = generate_weather_update()
air_quality_path, air_quality_n_new = generate_air_quality_update()

tripdata update: 293672 new rows, 49911 duplicate rows -> /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/raw/tripdata/tripdata_update_2024_04.parquet
weather update: 168 new rows, +humidity column -> /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/raw/weather/weather_update.csv


/var/folders/_d/2qd1ynb93rn1nn272s69lw640000gn/T/ipykernel_26481/2674189575.py:6: DtypeWarning: Columns (0: Qualifier) have mixed types. Specify dtype option on import or set low_memory=False.
  original = pd.read_csv(src_path)


air_quality update: 500 new rows, +aqi column -> /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/raw/air_quality/air_quality_update.csv


## Update files and expected schema changes

In [7]:
UPDATE_FILES = {
    "tripdata": "data/raw/tripdata/tripdata_update_2024_04.parquet",
    "weather": "data/raw/weather/weather_update.csv",
    "air_quality": "data/raw/air_quality/air_quality_update.csv",
}

SCHEMA_CHANGES = {
    "tripdata": "none",
    "weather": "added column 'humidity' (double, nullable)",
    "air_quality": "added column 'aqi' (int, nullable)",
}

## Ingest each update file and document the results

In [8]:
from src.common.config_loader import load_dataset_config
from src.common.spark_session import get_spark_session
from src.ingestion.ingestor import ingest_dataset

for dataset_name, rel_path in UPDATE_FILES.items():
    cfg = load_dataset_config(dataset_name)
    abs_path = os.path.join(PROJECT_ROOT, rel_path)

    results = ingest_dataset(spark, cfg, source_paths=[abs_path])
    for r in results:
        print(f"\n=== {dataset_name} ({r.source_file}) ===")
        print(f"  status:               {r.status}")
        print(f"  records read:         {r.records_read}")
        print(f"  new records inserted: {r.records_valid_silver}")
        print(f"  duplicates ignored:   {r.records_duplicate_ignored}")
        print(f"  records quarantined:  {r.records_quarantined}")
        print(f"  schema changes:       {SCHEMA_CHANGES[dataset_name]}")
        if r.error_message:
            print(f"  error: {r.error_message}")

26/09/24 21:49:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/09/24 21:49:38 WARN TaskSetManager: Stage 9 contains a task of very large size (53649 KiB). The maximum recommended task size is 1000 KiB.
26/09/24 21:49:43 WARN TaskSetManager: Stage 18 contains a task of very large size (53649 KiB). The maximum recommended task size is 1000 KiB.
26/09/24 21:49:50 WARN TaskSetManager: Stage 23 contains a task of very large size (53649 KiB). The maximum recommended task size is 1000 KiB.
26/09/24 21:49:55 WARN TaskSetManager: Stage 28 contains a task of very large size (53649 KiB). The maximum recommended task size is 1000 KiB.
                                                                                


=== tripdata (/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/raw/tripdata/tripdata_update_2024_04.parquet) ===
  status:               SUCCESS
  records read:         343583
  new records inserted: 293670
  duplicates ignored:   49911
  records quarantined:  2
  schema changes:       none

=== weather (/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/raw/weather/weather_update.csv) ===
  status:               SUCCESS
  records read:         168
  new records inserted: 168
  duplicates ignored:   0
  records quarantined:  0
  schema changes:       added column 'humidity' (double, nullable)


26/09/24 21:52:03 WARN TaskSetManager: Stage 74 contains a task of very large size (26466 KiB). The maximum recommended task size is 1000 KiB.
26/09/24 21:52:06 WARN TaskSetManager: Stage 83 contains a task of very large size (26466 KiB). The maximum recommended task size is 1000 KiB.
26/09/24 21:52:08 WARN TaskSetManager: Stage 92 contains a task of very large size (26466 KiB). The maximum recommended task size is 1000 KiB.
[Stage 92:>                                                       (0 + 12) / 12]


=== air_quality (/Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/raw/air_quality/air_quality_update.csv) ===
  status:               SUCCESS
  records read:         500
  new records inserted: 474
  duplicates ignored:   26
  records quarantined:  0
  schema changes:       added column 'aqi' (int, nullable)
